# Comparar experimentos BO
Rangos configurados en `optuna_study.db` y parámetros fijos de `PARAM.yml`.
Información no disponible: celda vacía. Si hubo distintos rangos dentro de un experimento, se muestran separados por `/`.

Las primeras filas muestran meses de entrenamiento, dataset, métrica registrada y mejor resultado de la BO.


In [17]:
import os
import json
import sqlite3
import pandas as pd
import yaml
from IPython.display import display
import rdata
import numpy as np


In [3]:
BASE_URL = os.path.join(os.path.expanduser("~"), "code", "DMEyF", "dmeyf2026")
# En la VM:
# BASE_URL = os.path.join(os.path.expanduser("~"), "buckets", "b1", "dmeyf2026")

EXP_URL = os.path.join(BASE_URL, "DATA", "EXP")

In [ ]:
experimentos = [
    "LAG-009"
]

In [ ]:
def mostrar_valor(valor):
    if isinstance(valor, float) and valor.is_integer():
        return str(int(valor))
    return str(valor)


def leer_experimento(experimento):
    carpeta = os.path.join(EXP_URL, experimento)
    archivo_db = os.path.join(carpeta, "optuna_study.db")
    archivo_param = os.path.join(carpeta, "PARAM.yml")

    if not os.path.isfile(archivo_db):
        return {}, {}

    # Solo lectura: no crea ni modifica la base.
    with sqlite3.connect(f"file:{os.path.abspath(archivo_db)}?mode=ro", uri=True) as conexion:
        distribuciones = conexion.execute(
            """
            SELECT DISTINCT p.param_name, p.distribution_json
            FROM trial_params AS p
            JOIN trials AS t ON t.trial_id = p.trial_id
            JOIN studies AS s ON s.study_id = t.study_id
            WHERE s.study_name = ?
            ORDER BY p.param_name
            """,
            (experimento,),
        ).fetchall()

    if not distribuciones:
        return {}, {}

    rangos = {}
    constantes = {}
    for nombre, distribucion_json in distribuciones:
        atributos = json.loads(distribucion_json)["attributes"]
        if "choices" in atributos:
            valores = atributos["choices"]
            variable = len(valores) > 1
            texto = " / ".join(mostrar_valor(v) for v in valores)
        else:
            minimo, maximo = atributos["low"], atributos["high"]
            variable = minimo != maximo
            texto = (
                f"min: {mostrar_valor(minimo)} | max: {mostrar_valor(maximo)}"
                if variable else mostrar_valor(minimo)
            )

        destino = rangos if variable else constantes
        destino.setdefault(nombre, [])
        if texto not in destino[nombre]:
            destino[nombre].append(texto)

    fijos = {}
    if os.path.isfile(archivo_param):
        with open(archivo_param, encoding="utf-8") as archivo:
            parametros = yaml.safe_load(archivo) or {}
        fijos = {
            nombre: mostrar_valor(valor)
            for nombre, valor in parametros.get("lgbm", {}).get("param_fijos", {}).items()
            if nombre not in rangos
        }

    # Un rango con mínimo = máximo es fijo. Su valor prevalece sobre PARAM.yml.
    for nombre, valores in constantes.items():
        if nombre not in rangos:
            fijos[nombre] = " / ".join(valores)

    rangos = {nombre: " / ".join(valores) for nombre, valores in rangos.items()}
    return rangos, fijos


def leer_resumen(experimento):
    carpeta = os.path.join(EXP_URL, experimento)
    archivo_param = os.path.join(carpeta, "PARAM.yml")
    archivo_db = os.path.join(carpeta, "optuna_study.db")
    parametros = {}
    if os.path.isfile(archivo_param):
        with open(archivo_param, encoding="utf-8") as archivo:
            parametros = yaml.safe_load(archivo) or {}

    configuracion = {}
    mejor = parametros.get("out", {}).get("lgbm", {}).get("y")
    if os.path.isfile(archivo_db):
        with sqlite3.connect(f"file:{os.path.abspath(archivo_db)}?mode=ro", uri=True) as conexion:
            atributo = conexion.execute(
                """
                SELECT a.value_json FROM study_user_attributes AS a
                JOIN studies AS s ON s.study_id = a.study_id
                WHERE s.study_name = ? AND a.key = 'configuracion_bo'
                """, (experimento,),
            ).fetchone()
            if atributo:
                configuracion = json.loads(atributo[0])
            resultado = conexion.execute(
                """
                SELECT CASE WHEN d.direction = 'MINIMIZE'
                       THEN MIN(v.value) ELSE MAX(v.value) END
                FROM trial_values AS v
                JOIN trials AS t ON t.trial_id = v.trial_id
                JOIN studies AS s ON s.study_id = t.study_id
                JOIN study_directions AS d ON d.study_id = s.study_id AND d.objective = 0
                WHERE s.study_name = ? AND t.state = 'COMPLETE' AND v.objective = 0
                """, (experimento,),
            ).fetchone()
            if resultado and resultado[0] is not None:
                mejor = resultado[0]

    meses = configuracion.get("train", parametros.get("train"))
    dataset = configuracion.get("dataset", parametros.get("dataset"))
    fijos = configuracion.get("param_fijos", parametros.get("lgbm", {}).get("param_fijos", {}))
    metrica = fijos.get("metric")
    if metrica == "average_precision":
        metrica = "ap"

    return {
        "Meses entrenamiento": ", ".join(map(str, meses)) if meses is not None else "",
        "Dataset": os.path.basename(dataset) if dataset else "",
        "Métrica": mostrar_valor(metrica) if metrica is not None else "",
        "Mejor resultado": mostrar_valor(mejor) if mejor is not None else "",
    }


resumen_por_experimento = {}
rangos_por_experimento = {}
fijos_por_experimento = {}
for experimento in experimentos:
    resumen_por_experimento[experimento] = leer_resumen(experimento)
    rangos, fijos = leer_experimento(experimento)
    rangos_por_experimento[experimento] = rangos
    fijos_por_experimento[experimento] = fijos

## HP variables

In [ ]:
tabla_variables = pd.DataFrame(rangos_por_experimento).reindex(columns=experimentos)
tabla_variables = tabla_variables.sort_index()
resumen = pd.DataFrame(resumen_por_experimento).reindex(columns=experimentos)
tabla_variables = pd.concat([resumen, tabla_variables]).fillna("")
tabla_variables.index.name = ""
display(tabla_variables)

,LAG-008,LAG-009,LAG-011,LAG-012,LAG-021,LAG-022
,,,,,,
Meses entrenamiento,202104,202104,"202103, 202104, 202105","202103, 202104, 202105","202103, 202104","202103, 202104"
Dataset,,,c_01_tern_2_lags_O2.csv,c_01_tern_2_lf_O2_k3_p6.csv,c_01_tern_2_lags.csv,c_01_tern_2_lf_k6_p4.csv
Métrica,ap,ap,auc,auc,ap,ap
Mejor resultado,0.13669331697916912,0.1324890378631483,0.937885756006709,0.9393748539850824,0.1340972730186387,0.13505078742437668
feature_fraction,min: 0.3 | max: 0.75,min: 0.1 | max: 0.35,min: 0.1 | max: 0.75,min: 0.1 | max: 0.75,min: 0.1 | max: 0.75,min: 0.1 | max: 0.75
learning_rate,min: 0.002 | max: 0.02,min: 0.005 | max: 0.025,min: 0.002 | max: 0.025,min: 0.002 | max: 0.025,min: 0.002 | max: 0.025,min: 0.002 | max: 0.025
min_data_in_leaf,min: 0 | max: 500,min: 100 | max: 1000,,,min: 0 | max: 1000,min: 0 | max: 1000
min_sum_hessian_in_leaf,min: 0.0001 | max: 0.1,min: 0.0001 | max: 0.1,min: 1e-06 | max: 0.1,min: 1e-06 | max: 0.1,min: 1e-06 | max: 0.1,min: 1e-06 | max: 0.1
num_iterations,min: 700 | max: 4000,min: 400 | max: 2000,min: 400 | max: 4000,min: 400 | max: 4000,min: 400 | max: 4000,min: 400 | max: 4000


## HP fijos

In [ ]:
tabla_fijos = pd.DataFrame(fijos_por_experimento).reindex(columns=experimentos)
tabla_fijos = tabla_fijos.sort_index().fillna("")
tabla_fijos.index.name = "HP"
display(tabla_fijos)

,LAG-009,LAG-011,LAG-012
HP,,,
bagging_fraction,1,1,1
boost_from_average,True,True,True
boosting,gbdt,gbdt,gbdt
deterministic,True,True,True
drop_rate,0.1,0.1,0.1
early_stopping,0,0,0
extra_trees,False,False,False
feature_fraction_bynode,,0.2,0.2
feature_pre_filter,False,False,False


# 1 experimetno, sus benchmarks

In [ ]:

experimento_buscado = "LAG-009"


def convertir_param(valor):
    if isinstance(valor, dict):
        return {k: convertir_param(v) for k, v in valor.items()}
    if isinstance(valor, np.ndarray):
        return convertir_param(
            valor.item() if valor.size == 1 else valor.tolist()
        )
    if isinstance(valor, (list, tuple)):
        return [convertir_param(v) for v in valor]
    if isinstance(valor, np.generic):
        return valor.item()
    return valor


def nombre_exp(valor):
    return str(valor).replace("\\", "/").rstrip("/").split("/")[-1]


def texto_bench(valor):
    if isinstance(valor, (list, tuple)):
        return ", ".join(texto_bench(v) for v in valor)
    return "" if valor is None else mostrar_valor(valor)


filas = []
for benchmark in sorted(os.listdir(EXP_URL)):
    if not benchmark.startswith("benchmark_multisemilla"):
        continue

    carpeta = os.path.join(EXP_URL, benchmark)
    archivo_param = os.path.join(carpeta, "PARAM.rds")
    archivo_modelos = os.path.join(carpeta, "mejor_corte_por_modelo.tsv")
    if not os.path.isfile(archivo_modelos):
        continue

    modelos = pd.read_csv(archivo_modelos, sep="\t")
    modelos = modelos.loc[
        modelos["experimento_origen"].map(nombre_exp).eq(experimento_buscado)
    ]
    if modelos.empty:
        continue

    param = (
        convertir_param(rdata.read_rds(archivo_param))
        if os.path.isfile(archivo_param) else {}
    )
    fijos = param.get("lgbm", {}).get("param_fijos", {})

    for _, modelo in modelos.iterrows():
        hp = fijos.copy()
        for nombre in fijos:
            if nombre in modelo and pd.notna(modelo[nombre]):
                hp[nombre] = modelo[nombre]

        # Ajuste que aplica el benchmark al entrenar.
        if "min_data_in_leaf" in hp:
            undersampling = param.get("trainingstrategy", {}).get("undersampling")
            hp["min_data_in_leaf"] = (
                round(hp["min_data_in_leaf"] / undersampling)
                if undersampling else None
            )
        if "semillas" in param:
            hp["seed"] = param["semillas"]

        filas.append({
            
            "Benchmark": benchmark.removeprefix("benchmark_multisemilla_lags"),
            "Ganancia": modelo.get("ganancia_total_promedio", ""),
            "Modelo": modelo.get("modelo_id", ""),
            "Dataset": param.get("dataset", modelo.get("dataset_bench", "")),
            "Meses": texto_bench(param.get("train_final", param.get("train"))),
            "Excluidas": texto_bench(param.get("columnas_excluidas")),
            "HP": hp,
        })

tabla_benchmarks = pd.DataFrame(
    filas, columns=["Benchmark", "Ganancia","Modelo", "Dataset", "Meses", "Excluidas", "HP"]
)

hp_por_modelo = pd.DataFrame([fila["HP"] for fila in filas]).map(
    lambda v: ""
    if v is None or (np.isscalar(v) and pd.isna(v))
    else texto_bench(v)
)
hp_distintos = [
    hp for hp in hp_por_modelo
    if hp_por_modelo.loc[hp_por_modelo[hp].ne(""), hp].nunique() > 1
]

tabla_benchmarks = pd.concat([
    tabla_benchmarks.drop(columns="HP"),
    hp_por_modelo[hp_distintos],
], axis=1)

display(tabla_benchmarks.sort_values(by="Benchmark"))

,Benchmark,Ganancia,Modelo,Dataset,Meses,Excluidas,min_data_in_leaf,feature_fraction,learning_rate,num_iterations,num_leaves,min_sum_hessian_in_leaf
0,20260924_090149,461010000,3,competencia_01_ternaria_lf_k6_perdida4.csv,202104,,1370,0.239049018582973,0.0152796385349906,400,256,0.1
1,20260924_090149,459470000,2,competencia_01_ternaria_lf_k6_perdida4.csv,202104,,1680,0.35,0.0067549325969674,1067,2048,0.0001
2,20260924_090149,455840000,1,competencia_01_ternaria_lf_k6_perdida4.csv,202104,,1810,0.264156247389931,0.0185711199192682,454,256,0.1
3,20260924_090643,469700000,2,competencia_01_ternaria_lags.csv,202104,,1680,0.35,0.0067549325969674,1067,2048,0.0001
4,20260924_090643,466730000,3,competencia_01_ternaria_lags.csv,202104,,1370,0.239049018582973,0.0152796385349906,400,256,0.1
5,20260924_090643,464640000,1,competencia_01_ternaria_lags.csv,202104,,1810,0.264156247389931,0.0185711199192682,454,256,0.1
6,20260924_114539,468600000,1,competencia_01_ternaria_lags.csv,"202103, 202104",,1810,0.264156247389931,0.0185711199192682,454,256,0.1
7,20260924_114539,467390000,2,competencia_01_ternaria_lags.csv,"202103, 202104",,1680,0.35,0.0067549325969674,1067,2048,0.0001
8,20260924_114539,467060000,3,competencia_01_ternaria_lags.csv,"202103, 202104",,1370,0.239049018582973,0.0152796385349906,400,256,0.1
9,20260924_123618,469920000,3,competencia_01_ternaria_lf_k6_perdida4.csv,"202103, 202104",,1370,0.239049018582973,0.0152796385349906,400,256,0.1
